# Chapter 30: Global Alignment

<a href="../lite/lab/index.html?path=ch30_global_alignment.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def draw_cov_ellipse(ax, mean, cov, n_std=2, **kwargs):
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1,1], vecs[0,1]))
    w, h = 2 * n_std * np.sqrt(np.maximum(vals, 0))
    ax.add_patch(Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs))

You have 4 maps of different rooms, each in its own coordinate frame. Stitch them together
wrong and doors connect to walls. Global alignment ensures every sub map agrees on where
North is and how everything fits together.

This chapter covers the math of aligning multiple maps into a single globally consistent frame.

## 30.1 Map Stitching

Given two point sets with known correspondences, the optimal rigid alignment (rotation + translation)
is found by minimizing:

$$\min_{R, t} \sum_i \| R p_i + t - q_i \|^2$$

The closed form solution uses SVD of the cross-covariance matrix.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
n_points = 8
true_angle = np.radians(25)
true_t = np.array([3.0, 1.5])
noise = 0.15
# ──────────────────────────────────────────────────────────────────────────────

R_true = np.array([[np.cos(true_angle), -np.sin(true_angle)],
                    [np.sin(true_angle), np.cos(true_angle)]])

# Source points
P = np.random.uniform(0, 5, (n_points, 2))
Q = (R_true @ P.T).T + true_t + np.random.normal(0, noise, (n_points, 2))

# SVD alignment
p_mean = P.mean(axis=0); q_mean = Q.mean(axis=0)
P_c = P - p_mean; Q_c = Q - q_mean
H = P_c.T @ Q_c
U, S, Vt = np.linalg.svd(H)
R_est = Vt.T @ U.T
if np.linalg.det(R_est) < 0:
    Vt[-1, :] *= -1
    R_est = Vt.T @ U.T
t_est = q_mean - R_est @ p_mean

P_aligned = (R_est @ P.T).T + t_est

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(P[:, 0], P[:, 1], c='steelblue', s=60, label='source P')
ax.scatter(Q[:, 0], Q[:, 1], c='tomato', s=60, label='target Q')
for i in range(n_points):
    ax.plot([P[i,0], Q[i,0]], [P[i,1], Q[i,1]], 'gray', alpha=0.3)
ax.set_title("Before alignment", fontsize=13); ax.set_aspect('equal'); ax.legend()

ax = axes[1]
ax.scatter(P_aligned[:, 0], P_aligned[:, 1], c='steelblue', s=60, label='aligned P')
ax.scatter(Q[:, 0], Q[:, 1], c='tomato', s=60, label='target Q')
for i in range(n_points):
    ax.plot([P_aligned[i,0], Q[i,0]], [P_aligned[i,1], Q[i,1]], 'forestgreen', alpha=0.5)
ax.set_title("After SVD alignment", fontsize=13); ax.set_aspect('equal'); ax.legend()

plt.tight_layout()
plt.show()

angle_est = np.degrees(np.arctan2(R_est[1,0], R_est[0,0]))
print(f"True:      angle = {np.degrees(true_angle):.1f}°, t = {true_t}")
print(f"Estimated: angle = {angle_est:.1f}°, t = {t_est}")
print(f"Alignment RMSE: {np.sqrt(np.mean(np.sum((P_aligned - Q)**2, axis=1))):.4f} m")

## 30.2 Frame Consistency

After alignment, verify consistency: transformed points from map A should coincide
with corresponding points in map B within the expected noise level.

**Key observations:**
- SVD alignment is the gold standard for rigid point set registration with known correspondences.
- It is a closed form solution (no iteration needed).
- Unknown correspondences require ICP (iterative closest point) which we cover in later chapters.

---

## Exercises

### Exercise 30.1
Implement SVD alignment for 3D point sets (3×3 rotation + 3D translation). Test on 20 random
3D points with a known rotation of 30° about the Z axis and translation [1, 2, 3].

In [ ]:
# Your code here